[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/okumura-kobe/kobe-data-business/blob/main/lecture06/lecture6_notebook.ipynb)

# 後期大学院演習「データで動かす社会とビジネス」
## 第6回　演習 Notebook

**テーマ：信用スコアリング：デフォルト確率を推定する**
― 情報が増えると、PDはどう変わるか ―

---

### この Notebook の使い方

- **STEP 1〜5 の順番に実行**してください
- 各 STEP の結果をワークシートに記録してください
- 授業中は STEP 0〜5 の完成コードを実行し、分析結果をワークシートに記録・解釈します。授業後は末尾の **Coding Challenge** で、Logit 推定と PD 算出を自分で再現してください。Coding Challenge は**提出不要・採点対象外**です

> ⚠️ **今回行うこと**：Model A / B の PD を作る（「作る」回）
> ⚠️ **今回行わないこと**：AUC・Calibration・Cut-off・Model A/B の優劣判断（→第7回）

**今回の中心的な問い**
> 新しい情報を加えると、PD はどう変わるのか。

---

| | |
|---|---|
| 学籍番号 | |
| 氏名 | |
| 実施日 | |

---
## Step 0　準備

In [ ]:
# ライブラリの読み込み（変更不要）
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['figure.figsize']    = (11, 4)

print('Ready')

In [ ]:
# データの読み込み（変更不要）
# credit_model.csv：Development / Validation 区分が既に設定済み
url = "https://raw.githubusercontent.com/okumura-kobe/kobe-data-business/main/data/credit_model.csv"
df = pd.read_csv(url)

print(f'総件数: {len(df):,}件　変数数: {df.shape[1]}列')
print(f'列名: {df.columns.tolist()}')

---
## STEP 1　Development / Validation を確認する（10分）

第6回では、**既に固定された区分**を使います。自分で再分割しないでください。

- Development：7,000件　→　Model A / B の係数推定に使用
- Validation　：3,000件　→　**第7回**のモデル検証に使用（今回は使わない）

これにより、第6〜9回の結果を学生間で比較可能にします。

In [ ]:
# 1-1　Development / Validation の件数と Default率を確認する
# ⚠️ 再分割しない。split 列を使って区分する。

dev = df[df['split'] == 'development'].copy().reset_index(drop=True)
val = df[df['split'] == 'validation'].copy().reset_index(drop=True)

print('=== Development ===')
print(f'件数: {len(dev):,}件')
print(f'Default件数: {dev["default"].sum():,}件')
print(f'Default率: {dev["default"].mean():.4f}')

print()
print('=== Validation ===')
print(f'件数: {len(val):,}件')
print(f'Default件数: {val["default"].sum():,}件')
print(f'Default率: {val["default"].mean():.4f}')

print()
print('=== 確認 ===')
print(f'合計: {len(dev)+len(val):,}件（元データ: {len(df):,}件）')

---
## STEP 2　Model A を推定する（10分）

**Model A（Current Model）**：直近時点の情報を中心とした5変数

| 変数 | 意味 |
|---|---|
| `age` | 年齢 |
| `credit_limit` | 与信限度額 |
| `latest_payment_status` | 直近返済ステータス |
| `latest_bill_amount` | 直近請求額 |
| `latest_payment_amount` | 直近支払額 |

Logit（ロジスティック回帰）で PD を推定します。
$$\log\left(\frac{p}{1-p}\right) = \beta_0 + \beta_1 x_1 + \cdots + \beta_k x_k$$

In [ ]:
# 2-1　Model A の変数を定義する
features_A = [
    "age",
    "credit_limit",
    "latest_payment_status",
    "latest_bill_amount",
    "latest_payment_amount"
]

print(f'Model A 変数数: {len(features_A)}変数')
print(features_A)

In [ ]:
# 2-2　Model A を推定する（Development sample）
X_A = sm.add_constant(dev[features_A])

model_A = sm.Logit(
    dev["default"],
    X_A
).fit()

print(model_A.summary())

In [ ]:
# 2-3　Development sample に pd_A を作成する
dev["pd_A"] = model_A.predict(X_A)

print('=== pd_A の基本統計 ===')
print(dev["pd_A"].describe().round(4))

---
## STEP 3　Model B を推定する（10分）

**Model B（Candidate Model）**：Model A の5変数 ＋ 履歴6変数 ＝ 11変数

| 追加する変数 | 意味 |
|---|---|
| `max_delay` | 延滞の深刻度（過去6か月の最大延滞ステータス） |
| `months_with_delay` | 延滞の頻度（過去6か月の延滞月数） |
| `avg_bill_amount` | 過去6か月平均請求額 |
| `avg_payment_amount` | 過去6か月平均支払額 |
| `credit_utilization` | 与信利用率 |
| `payment_ratio` | 支払比率 |

Model A と同じ Logit で推定します。**変えるのは投入する情報だけです。**

In [ ]:
# 3-1　Model B の変数を定義する（Model A の変数をすべて引き継ぐ）
features_B = features_A + [
    "max_delay",
    "months_with_delay",
    "avg_bill_amount",
    "avg_payment_amount",
    "credit_utilization",
    "payment_ratio"
]

print(f'Model B 変数数: {len(features_B)}変数')
print(f'  Model A: {len(features_A)}変数 ＋ 履歴特徴量: {len(features_B)-len(features_A)}変数')

In [ ]:
# 3-2　Model B を推定する（Development sample）
X_B = sm.add_constant(dev[features_B])

model_B = sm.Logit(
    dev["default"],
    X_B
).fit()

print(model_B.summary())

In [ ]:
# 3-3　Development sample に pd_B を作成する
dev["pd_B"] = model_B.predict(X_B)

print('=== pd_B の基本統計 ===')
print(dev["pd_B"].describe().round(4))

---
## STEP 4　PD を比較する（10分）

同じ顧客について `pd_A` と `pd_B` を比較します。

- PD が**上昇した顧客**（Model B がより高リスクと見る）
- PD が**低下した顧客**（Model B がより低リスクと見る）
- **ほとんど変化しなかった顧客**

を確認してください。

> ⚠️ PD が変わった ≠ Model B が改善した。変化の意味は第7回で検証します。

In [ ]:
# 4-1　pd_A と pd_B の差分を計算する
dev["pd_diff"] = dev["pd_B"] - dev["pd_A"]

print('=== pd_diff（pd_B - pd_A）の統計 ===')
print(dev["pd_diff"].describe().round(4))

print()
# 変化の方向別に分類
up   = (dev["pd_diff"] >  0.05).sum()
down = (dev["pd_diff"] < -0.05).sum()
flat = ((dev["pd_diff"] >= -0.05) & (dev["pd_diff"] <= 0.05)).sum()
print(f'大きく上昇（pd_diff > +0.05）: {up:,}件')
print(f'大きく低下（pd_diff < -0.05）: {down:,}件')
print(f'ほぼ変化なし（pd_diff が ±0.05 以内）: {flat:,}件')

In [ ]:
# 4-2　pd_A vs pd_B の散布図
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(dev["pd_A"], dev["pd_B"], alpha=0.1, s=5, color='#1E2761')
ax.plot([0, 1], [0, 1], color='#C00000', linestyle='--', lw=1.5, label='pd_A = pd_B')
ax.set_xlabel('PD from Model A')
ax.set_ylabel('PD from Model B')
ax.set_title('Model A vs Model B: Predicted PD (Development Sample)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 4-3　PD が大きく変化した顧客を確認する
cols = ["customer_id", "default", "pd_A", "pd_B", "pd_diff",
        "latest_payment_status", "max_delay", "months_with_delay",
        "credit_utilization", "payment_ratio"]

print('=== pd_B が大きく上昇した顧客（上位10件）===')
print(dev.nlargest(10, "pd_diff")[cols].to_string(index=False))

print()
print('=== pd_B が大きく低下した顧客（低下幅の大きい10件）===')
print(dev.nsmallest(10, "pd_diff")[cols].to_string(index=False))

---
## STEP 5　変化を解釈する（10分）

PD が大きく変化した顧客について、履歴特徴量の値を確認し、
**なぜ Model B で PD が変わったと考えられるか**を整理してください。

ワークシートの **STEP 5** に記録してください。

> ⚠️ 特定変数の因果効果を主張しないでください。
> あくまで「この特徴量の値が、このモデルではこのように効いている」という解釈に留めてください。

In [ ]:
# 5-1　PD が上昇した顧客の特徴を確認する
hist_vars = ["max_delay", "months_with_delay", "avg_bill_amount",
             "avg_payment_amount", "credit_utilization", "payment_ratio"]

# PD上昇幅上位 vs PD低下幅上位 vs 全体の平均比較
top_up   = dev.nlargest(100, "pd_diff")
top_down = dev.nsmallest(100, "pd_diff")

print('=== 履歴特徴量の平均（PD上昇上位100件 vs PD低下上位100件 vs 全体）===')
compare = pd.DataFrame({
    'PD上昇（+）': top_up[hist_vars].mean(),
    'PD低下（−）': top_down[hist_vars].mean(),
    '全体平均':    dev[hist_vars].mean(),
}).round(3)
print(compare)

In [ ]:
# 5-2　解釈のメモ（コメントに記録してください）
#
# 【PD が上昇した顧客の特徴】
# （max_delay・months_with_delay などが高い傾向があるか？）
#
#
# 【PD が低下した顧客の特徴】
# （どの履歴特徴量に特徴的な違いが見られるか？）
#
#
# 【Model B で PD が変わった理由の解釈】
# （Model A にない情報がどう機能したか）
#
#


---
## まとめ

### 今日作ったもの

| 項目 | 内容 |
|---|---|
| Model A | 5変数 Logit、Development で推定済み |
| Model B | 11変数 Logit、Development で推定済み |
| pd_A | Development 7,000件の顧客別 PD（Model A） |
| pd_B | Development 7,000件の顧客別 PD（Model B） |
| pd_diff | pd_B - pd_A（PD の変化） |

### 今回の核心

> **同じ顧客でも、見る情報でリスクの見え方は変わる。**

> **しかし、PD が変わったことと、モデルが改善したことは同じではない。**

---

### 第7回の予告：PD を確かめる

第7回では、今日作った Model A / B を **Validation sample（3,000件）** に適用します。

- AUC（判別力）
- Calibration（PD の精度）
- Cut-off 別の Confusion Matrix

を使って、**本当にどちらが良いモデルか**を検証します。

**事前課題**
- 今日の結果（pd_A・pd_B の比較・変化の解釈）をワークシートに記録しておく
- BEEF+ の第7回配布資料を読む

---
## Coding Challenge：Model A の Logit 推定と顧客別 PD 算出を自分で再現する

> **授業外の自習課題です。提出不要・採点対象外です。**（想定時間：30〜45分）
> 授業内の STEP 0〜5 の実行には影響しません。

授業中は完成コードを実行して、分析結果を解釈しました。ここでは、STEP 2 で行った **Model A の推定と PD の算出** を、自分でコードを書いて再現します。

**前提**：STEP 0〜1 を実行して、Development sample の `dev` を作成済みであること（`dev` は 7,000 件）。

### 課題

`dev` を使い、次の処理を自分で記述してください。

1. Model A の5つの説明変数を指定する（`features_cc`）
2. 切片を追加する（`X_cc`）
3. `default` を目的変数として Logit モデルを推定する（`model_cc`）
4. Development sample の各顧客について PD を算出する（`pd_cc`）
5. 算出した PD の平均値・最小値・最大値を表示する

> 授業内の `model_A`・`X_A`・`pd_A` を上書きしないよう、変数名は上の括弧内の名前を使ってください。

### ヒント

- `statsmodels.api`（`sm`）は読み込み済みです
- `sm.add_constant()`：切片（定数項）を追加する
- `sm.Logit()` と `.fit()`：Logit モデルを推定する
- `.predict()`：推定したモデルで PD を算出する
- `.mean()`・`.min()`・`.max()`：平均値・最小値・最大値

授業内の STEP 2 のコードを見ながら、自分の言葉でコードを組み立ててみてください。

In [ ]:
# Coding Challenge：ここに自分でコードを書いてください
# （変数名：features_cc / X_cc / model_cc / pd_cc）

# 1. Model A の5つの説明変数を指定する


# 2. 切片を追加する


# 3. default を目的変数として Logit を推定する


# 4. 各顧客の PD を算出する


# 5. PD の平均値・最小値・最大値を表示する


### 自己確認

上のセルで `model_cc` と `pd_cc` を作ってから、次のセルを実行してください。次の5点を確認します。

- 推定に使用した件数が 7,000 件である
- 説明変数が 5 変数（切片を除く）である
- 予測 PD の件数が 7,000 件である
- 平均 PD が、Development sample の実績 Default 率と概ね一致する
- 推定 PD がすべて 0〜1 の範囲内にある

> 平均 PD が実績 Default 率と一致するのは、**切片を含む Logit モデルを Development sample 自身に当てはめたときの性質**です。Validation sample（第7回）では、一致するとは限りません。

In [ ]:
# Coding Challenge 自己確認（変更不要）
# model_cc と pd_cc を作ってから実行してください
try:
    names  = [p for p in model_cc.params.index if p != 'const']
    n_obs  = int(model_cc.nobs)
    n_x    = len(names)
    has_c  = 'const' in model_cc.params.index
    pd_arr = np.asarray(pd_cc, dtype=float)
    n_pd   = len(pd_arr)
    mean_pd = pd_arr.mean()
    dev_dr  = dev['default'].mean()
    in_rng  = bool((pd_arr >= 0).all() and (pd_arr <= 1).all())
except NameError:
    print('model_cc または pd_cc がまだありません。上のセルで作成してから実行してください。')
else:
    def mark(ok): return 'OK' if ok else '要確認'
    print(f'推定に使用した件数    : {n_obs:,}件（期待値 7,000）  → {mark(n_obs == 7000)}')
    print(f'説明変数（切片を除く）: {n_x}変数（期待値 5）・切片あり={has_c}  → {mark(n_x == 5 and has_c)}')
    print(f'予測PDの件数          : {n_pd:,}件（期待値 7,000）  → {mark(n_pd == 7000)}')
    print(f'平均PD {mean_pd:.4f} / 実績Default率 {dev_dr:.4f}  → {mark(abs(mean_pd - dev_dr) < 0.001)}')
    print(f'PDの範囲 {pd_arr.min():.4f}〜{pd_arr.max():.4f}（0〜1の範囲内）  → {mark(in_rng)}')
    # 参考：授業内の Model A（STEP 2）を実行済みなら、PD を比較する
    if 'model_A' in globals() and 'X_A' in globals() and n_pd == 7000:
        gap = np.abs(pd_arr - np.asarray(model_A.predict(X_A))).max()
        print(f'（参考）授業内の Model A の PD との最大差: {gap:.6f}')


---
### 解答例（自習後に確認してください）

**まず自分でコードを書いてから**、次のセルを開いてください。セルは折りたたまれています。Colab では、セルの「…」メニューの「コードを表示」で開けます。

解答例は、授業内の `model_A`・`X_A`・`pd_A` を使わず、別の変数名で Model A を最初から推定し直しています。

In [ ]:
#@title 解答例（自習後に確認する） { display-mode: "form" }
# 授業内の model_A / X_A / pd_A は使わず、別の変数名で Model A を最初から推定し直す

# 1. Model A の5つの説明変数を指定する
features_sol = [
    "age",
    "credit_limit",
    "latest_payment_status",
    "latest_bill_amount",
    "latest_payment_amount"
]

# 2. 切片を追加する
X_sol = sm.add_constant(dev[features_sol])

# 3. default を目的変数として Logit を推定する
model_sol = sm.Logit(dev["default"], X_sol).fit()

# 4. 各顧客の PD を算出する
pd_sol = model_sol.predict(X_sol)

# 5. PD の平均値・最小値・最大値を表示する
print()
print(f'平均PD: {pd_sol.mean():.4f}')
print(f'最小PD: {pd_sol.min():.4f}')
print(f'最大PD: {pd_sol.max():.4f}')

# 確認：件数・説明変数の数・実績Default率との比較
print()
print(f'推定に使用した件数: {int(model_sol.nobs):,}件')
print(f'説明変数（切片を除く）: {len(features_sol)}変数')
print(f'予測PDの件数: {len(pd_sol):,}件')
print(f'実績Default率: {dev["default"].mean():.4f}')
